In [3]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

REQUEST_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}


def read_source_links(txt_path: str) -> list:
    """
    Reads metric page URLs from a text file.
    """
    file = Path(txt_path)
    if not file.exists():
        return []

    return [
        line.strip()
        for line in file.read_text(encoding="utf-8").splitlines()
        if line.strip().startswith("http")
    ]


def fetch_country_master() -> pd.DataFrame:
    """
    Extracts country names and global ranks from the main listing page.
    """
    url = "https://www.globalfirepower.com/countries-listing.php"
    html = requests.get(url, headers=REQUEST_HEADERS, timeout=20).text
    soup = BeautifulSoup(html, "lxml")

    records = []

    for container in soup.find_all("div", class_="recordsetContainer"):
        try:
            country = container.select_one("span.textShadow").get_text(strip=True)
            rank = container.select_one("span.textBold").get_text(strip=True)
            records.append({"Country": country, "Global_Rank": rank})
        except AttributeError:
            continue

    return pd.DataFrame(records)


def parse_metric_page(metric_url: str) -> pd.DataFrame:
    """
    Parses a single metric page and returns Country + Metric column.
    """
    response = requests.get(metric_url, headers=REQUEST_HEADERS, timeout=20)
    soup = BeautifulSoup(response.text, "lxml")

    metric_key = (
        metric_url.rsplit("/", 1)[-1]
        .replace(".php", "")
        .replace("-", "_")
    )

    rows = []

    for row in soup.find_all("div", class_="recordsetContainer"):
        spans = row.find_all("span", class_="textLarge")
        if len(spans) < 2:
            continue

        try:
            country = row.select_one("span.textShadow").get_text(strip=True)
            value = spans[-1].get_text(strip=True)
            rows.append({"Country": country, metric_key: value})
        except AttributeError:
            continue

    return pd.DataFrame(rows)


def normalize_numeric_fields(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans numeric columns by removing commas and extracting numbers.
    """
    for col in df.columns:
        if col in ("Country", "Global_Rank"):
            continue

        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.extract(r"(\d+(?:\.\d+)?)")[0]
        )

    return df


def build_global_firepower_dataset(url_file: str) -> pd.DataFrame:
    """
    Master pipeline:
    - Load country base
    - Loop metric URLs
    - Merge all metrics
    """
    master_df = fetch_country_master()
    metric_urls = read_source_links(url_file)

    for metric_url in metric_urls:
        try:
            metric_df = parse_metric_page(metric_url)
            if not metric_df.empty:
                master_df = master_df.merge(metric_df, on="Country", how="left")
        except Exception:
            continue

    return normalize_numeric_fields(master_df)

dataset = build_global_firepower_dataset("/content/All_links_for_golbal_military_data.txt")
dataset.to_csv("military_raw_data.csv", index=False)
print("military_raw_data.csv generated successfully")

military_raw_data.csv generated successfully


# Task
Map and rename the columns in the `dataset` DataFrame to align with the mandatory column names, using the provided mapping. After renaming, re-run the existing column check logic to evaluate the updated column presence.

Here is the renaming dictionary to use:
```python
rename_mapping = {
    'aircraft_total': 'total_aircraft',
    'aircraft_total_fighters': 'fighter_aircraft',
    'aircraft_total_attack_types': 'attack_aircraft',
    'aircraft_total_transports': 'transport_aircraft',
    'aircraft_helicopters_total': 'helicopters',
    'armor_tanks_total': 'tanks',
    'armor_apc_total': 'armored_vehicles',
    'armor_towed_artillery_total': 'artillery_units',
    'navy_ships': 'naval_assets',
    'navy_aircraft_carriers': 'aircraft_carriers',
    'defense_spending_budget': 'defense_budget_usd',
    'total_population_by_country': 'population',
    'square_land_area': 'land_area_sq_km',
    'coastline_coverage': 'coastline_km',
    'Global_Rank': 'power_index_rank',
    'active_military_manpower': 'active_personnel',
    'active_reserve_military_manpower': 'reserve_personnel',
    'available_military_manpower': 'total_personnel',
    'purchasing_power_parity': 'gdp_usd'
}
```

In [31]:
dataset.to_csv("military_raw_data.csv", index=False)
print("military_raw_data.csv generated successfully with updated data.")

military_raw_data.csv generated successfully with updated data.


In [34]:
import requests

REST_COUNTRIES_API_BASE = "https://restcountries.com/v3.1/name/"

def debug_fetch_full_country_data_from_restcountries(country_name: str):
    """
    Fetches full country data from REST Countries API for a given country name
    to debug unexpected results.
    """
    country_lookup_name = country_name.replace(' ', '%20') # URL encode spaces
    url = f"{REST_COUNTRIES_API_BASE}{country_lookup_name}"

    print(f"Fetching full data for {country_name} from: {url}")

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()

        if data:
            print(f"Full API response for {country_name}:")
            # Print only the first entry for brevity
            for key, value in data[0].items():
                print(f"  {key}: {value}")
        else:
            print(f"No data found for {country_name} from REST Countries API.")
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data for {country_name} from REST Countries API: {e}")

debug_fetch_full_country_data_from_restcountries('United States')


Fetching full data for United States from: https://restcountries.com/v3.1/name/United%20States
Full API response for United States:
  name: {'common': 'United States Virgin Islands', 'official': 'Virgin Islands of the United States', 'nativeName': {'eng': {'official': 'Virgin Islands of the United States', 'common': 'United States Virgin Islands'}}}
  tld: ['.vi']
  cca2: VI
  ccn3: 850
  cioc: ISV
  independent: False
  status: officially-assigned
  unMember: False
  currencies: {'USD': {'symbol': '$', 'name': 'United States dollar'}}
  idd: {'root': '+1', 'suffixes': ['340']}
  capital: ['Charlotte Amalie']
  altSpellings: ['VI', 'Virgin Islands, U.S.']
  region: Americas
  subregion: Caribbean
  languages: {'eng': 'English'}
  latlng: [18.35, -64.933333]
  landlocked: False
  area: 347.0
  demonyms: {'eng': {'f': 'Virgin Islander', 'm': 'Virgin Islander'}, 'fra': {'f': '', 'm': ''}}
  cca3: VIR
  translations: {'ara': {'official': 'جزر العذراء الامريكية', 'common': 'جزر العذراء الام

In [35]:
import requests

# Base URL for REST Countries API
REST_COUNTRIES_API_BASE = "https://restcountries.com/v3.1/name/"

def fetch_country_data_from_restcountries(country_name: str) -> dict:
    """
    Fetches country data from REST Countries API for a given country name.
    Extracts capital, region, and continent.
    """
    # REST Countries API handles common country names well
    country_lookup_name = country_name.replace(' ', '%20') # URL encode spaces
    # Add fullText=true to ensure exact match for the country name
    url = f"{REST_COUNTRIES_API_BASE}{country_lookup_name}?fullText=true&fields=capital,region,continents"

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        data = response.json()

        if data:
            # The API returns a list of countries, even for a single match, but with fullText=true it should be the correct one
            country_info = data[0]
            return {
                'Country': country_name,
                'capital_city': country_info['capital'][0] if 'capital' in country_info and country_info['capital'] else None,
                'region': country_info['region'] if 'region' in country_info else None,
                'continent': country_info['continents'][0] if 'continents' in country_info and country_info['continents'] else None
            }
        else:
            print(f"No data found for {country_name} from REST Countries API.")
            return {'Country': country_name, 'capital_city': None, 'region': None, 'continent': None}
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data for {country_name} from REST Countries API: {e}")
        return {'Country': country_name, 'capital_city': None, 'region': None, 'continent': None}

# Test with 'United States' again to confirm the fix
sample_country_data_rest = fetch_country_data_from_restcountries('United States')
print("Sample data from REST Countries API for United States (after fix):")
print(sample_country_data_rest)

# Test with 'Vatican City' again to ensure it still works
sample_country_data_rest_2 = fetch_country_data_from_restcountries('Vatican City')
print("\nSample data from REST Countries API for Vatican City (after fix):")
print(sample_country_data_rest_2)

Sample data from REST Countries API for United States (after fix):
{'Country': 'United States', 'capital_city': 'Washington, D.C.', 'region': 'Americas', 'continent': 'North America'}

Sample data from REST Countries API for Vatican City (after fix):
{'Country': 'Vatican City', 'capital_city': 'Vatican City', 'region': 'Europe', 'continent': 'Europe'}


In [36]:
from tqdm.notebook import tqdm
import datetime
import pandas as pd

# Prepare to collect details for all countries from REST Countries API
rest_country_details_list = []

# Iterate through each unique country in the dataset
for country in tqdm(dataset['Country'].unique(), desc="Fetching country details from REST Countries API"):
    try:
        details = fetch_country_data_from_restcountries(country)
        rest_country_details_list.append(details)
    except Exception as e:
        print(f"Error fetching data for {country} from REST Countries API: {e}")
        rest_country_details_list.append({'Country': country, 'capital_city': None, 'region': None, 'continent': None})

# Convert the list of dictionaries to a DataFrame
rest_country_details_df = pd.DataFrame(rest_country_details_list)

# Drop the old 'capital_city', 'region', 'continent', 'alliance', and 'year' columns if they exist before merging
# We are using REST Countries for capital_city, region, continent. Alliance and year are from other sources/logic.
dataset = dataset.drop(columns=['capital_city', 'region', 'continent', 'alliance', 'year'], errors='ignore')

# Merge the new details from REST Countries API into the main dataset
dataset = dataset.merge(rest_country_details_df, on='Country', how='left')

# Re-add the 'year' column with the current year (as it's not from REST Countries API)
current_year = datetime.datetime.now().year
dataset['year'] = current_year

# The 'alliance' column is not available directly from REST Countries API and was unreliable from globalfirepower.com
# Initialize it to None for all countries, or re-attempt scraping from another source if one is identified later.
if 'alliance' not in dataset.columns:
    dataset['alliance'] = None

print(f"Successfully fetched and merged capital city, region, continent, and added year ({current_year}) for {len(rest_country_details_df)} countries.")

Fetching country details from REST Countries API:   0%|          | 0/145 [00:00<?, ?it/s]

Error fetching data for Turkiye from REST Countries API: 404 Client Error: Not Found for url: https://restcountries.com/v3.1/name/Turkiye?fullText=true&fields=capital,region,continents
Error fetching data for Beliz from REST Countries API: 404 Client Error: Not Found for url: https://restcountries.com/v3.1/name/Beliz?fullText=true&fields=capital,region,continents
Successfully fetched and merged capital city, region, continent, and added year (2026) for 145 countries.


In [38]:
import numpy as np

columns_to_check = ['capital_city', 'region', 'continent']

print("--- Missing Value Analysis ---")

for col in columns_to_check:
    missing_count = dataset[col].isnull().sum()
    print(f"\nColumn '{col}':")
    print(f"  Total missing values: {missing_count}")

    if missing_count > 0:
        countries_with_missing = dataset[dataset[col].isnull()]['Country'].tolist()
        print(f"  Countries with missing '{col}': {countries_with_missing}")
    else:
        print(f"  No missing values for '{col}'.")


--- Missing Value Analysis ---

Column 'capital_city':
  Total missing values: 2
  Countries with missing 'capital_city': ['Turkiye', 'Beliz']

Column 'region':
  Total missing values: 2
  Countries with missing 'region': ['Turkiye', 'Beliz']

Column 'continent':
  Total missing values: 2
  Countries with missing 'continent': ['Turkiye', 'Beliz']


In [39]:
import requests
import pandas as pd
import datetime
from tqdm.notebook import tqdm

# Base URL for REST Countries API
REST_COUNTRIES_API_BASE = "https://restcountries.com/v3.1/name/"

# Mapping for problematic country names to API-friendly alternatives
COUNTRY_NAME_MAPPING = {
    'Turkiye': 'Turkey',
    'Beliz': 'Belize'
}

def fetch_country_data_from_restcountries(country_name: str) -> dict:
    """
    Fetches country data from REST Countries API for a given country name.
    Includes retry logic for known problematic country names.
    Extracts capital, region, and continent.
    """
    potential_country_names = [country_name]
    if country_name in COUNTRY_NAME_MAPPING:
        potential_country_names.append(COUNTRY_NAME_MAPPING[country_name])

    for current_name in potential_country_names:
        country_lookup_name = current_name.replace(' ', '%20')  # URL encode spaces
        url = f"{REST_COUNTRIES_API_BASE}{country_lookup_name}?fullText=true&fields=capital,region,continents"

        try:
            response = requests.get(url, timeout=10)
            response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
            data = response.json()

            if data:
                country_info = data[0]
                return {
                    'Country': country_name, # Return original country name for merging
                    'capital_city': country_info['capital'][0] if 'capital' in country_info and country_info['capital'] else None,
                    'region': country_info['region'] if 'region' in country_info else None,
                    'continent': country_info['continents'][0] if 'continents' in country_info and country_info['continents'] else None
                }
        except requests.exceptions.RequestException as e:
            # If it's a 404 and we have an alternative name, try the next one
            if response.status_code == 404 and current_name != potential_country_names[-1]:
                continue # Try the next alternative name
            else:
                print(f"Error fetching data for {current_name} (original: {country_name}) from REST Countries API: {e}")
                return {'Country': country_name, 'capital_city': None, 'region': None, 'continent': None}

    # If all attempts fail
    return {'Country': country_name, 'capital_city': None, 'region': None, 'continent': None}

# Prepare to collect details for all countries from REST Countries API
rest_country_details_list = []

# Iterate through each unique country in the dataset
for country in tqdm(dataset['Country'].unique(), desc="Fetching country details from REST Countries API"):
    try:
        details = fetch_country_data_from_restcountries(country)
        rest_country_details_list.append(details)
    except Exception as e:
        print(f"An unexpected error occurred while processing {country}: {e}")
        rest_country_details_list.append({'Country': country, 'capital_city': None, 'region': None, 'continent': None})

# Convert the list of dictionaries to a DataFrame
rest_country_details_df = pd.DataFrame(rest_country_details_list)

# Drop existing 'capital_city', 'region', 'continent', 'alliance', and 'year' columns from main dataset
# The 'alliance' and 'year' columns will be re-added or handled separately after merging.
dataset = dataset.drop(columns=['capital_city', 'region', 'continent', 'alliance', 'year'], errors='ignore')

# Merge the new details from REST Countries API into the main dataset
dataset = dataset.merge(rest_country_details_df, on='Country', how='left')

# Re-add the 'year' column with the current year (as it's not from REST Countries API)
current_year = datetime.datetime.now().year
dataset['year'] = current_year

# The 'alliance' column is not available directly from REST Countries API and was unreliable from globalfirepower.com
# Initialize it to None for all countries, or re-attempt scraping from another source if one is identified later.
if 'alliance' not in dataset.columns:
    dataset['alliance'] = None

print(f"Successfully fetched and merged capital city, region, continent, and added year ({current_year}) for {len(rest_country_details_df)} countries.")


Fetching country details from REST Countries API:   0%|          | 0/145 [00:00<?, ?it/s]

Successfully fetched and merged capital city, region, continent, and added year (2026) for 145 countries.


In [40]:
import numpy as np

columns_to_check = ['capital_city', 'region', 'continent']

print("--- Missing Value Analysis ---")

for col in columns_to_check:
    missing_count = dataset[col].isnull().sum()
    print(f"\nColumn '{col}':")
    print(f"  Total missing values: {missing_count}")

    if missing_count > 0:
        countries_with_missing = dataset[dataset[col].isnull()]['Country'].tolist()
        print(f"  Countries with missing '{col}': {countries_with_missing}")
    else:
        print(f"  No missing values for '{col}'.")

--- Missing Value Analysis ---

Column 'capital_city':
  Total missing values: 0
  No missing values for 'capital_city'.

Column 'region':
  Total missing values: 0
  No missing values for 'region'.

Column 'continent':
  Total missing values: 0
  No missing values for 'continent'.


**Reasoning**:
To standardize the scraped NATO member countries to match the naming convention in the `dataset` DataFrame, I will first define a mapping for problematic country names and then apply this mapping to the scraped `nato_member_countries` set. Afterward, I will define a function to categorize countries as 'NATO' or 'Other' based on this standardized set and apply it to the `dataset` DataFrame.



In [46]:
import pandas as pd

# 1. Define a dictionary to map French NATO member names to English equivalents
#    Based on the previous output: {'Canada', 'Bulgarie', ..., 'Etats-Unis', ...}
country_name_standardization_map = {
    'Etats-Unis': 'United States',
    'Royaume-Uni': 'United Kingdom',
    'La République de Macédoine du Nord': 'North Macedonia',
    'Belgique': 'Belgium',
    'Bulgarie': 'Bulgaria',
    'Tchéquie': 'Czechia',
    'Danemark': 'Denmark',
    'Estonie': 'Estonia',
    'Finlande': 'Finland',
    'France': 'France',
    'Allemagne': 'Germany',
    'Grèce': 'Greece',
    'Hongrie': 'Hungary',
    'Islande': 'Iceland',
    'Italie': 'Italy',
    'Lettonie': 'Latvia',
    'Lituanie': 'Lithuania',
    'Luxembourg': 'Luxembourg',
    'Monténégro': 'Montenegro',
    'Pays-Bas': 'Netherlands',
    'Norvège': 'Norway',
    'Pologne': 'Poland',
    'Portugal': 'Portugal',
    'Roumanie': 'Romania',
    'Slovaquie': 'Slovakia',
    'Slovénie': 'Slovenia',
    'Espagne': 'Spain',
    'Suède': 'Sweden',
    'Türkiye': 'Turkey' # Note: 'Turkiye' is already in the dataset
}

# Standardize the scraped NATO member countries
standardized_nato_members = set()
for member in nato_member_countries:
    standardized_nato_members.add(country_name_standardization_map.get(member, member))

print(f"Original scraped NATO members: {nato_member_countries}")
print(f"Standardized NATO members: {standardized_nato_members}")

# 2. Create a Python function to categorize countries
def get_alliance_status(country_name: str, nato_members_set: set) -> str:
    """
    Categorizes a country as 'NATO' or 'Other'.
    """
    if country_name in nato_members_set:
        return 'NATO'
    else:
        return 'Other'

# 3. Apply this function to the 'Country' column of the dataset DataFrame
dataset['alliance'] = dataset['Country'].apply(lambda x: get_alliance_status(x, standardized_nato_members))

print(f"Updated 'alliance' column for first 5 rows:\n{dataset[['Country', 'alliance']].head()}")

Original scraped NATO members: {'Canada', 'Bulgarie', 'Luxembourg', 'Royaume-Uni', 'Tchéquie', 'Suède', 'Pays-Bas', 'Hongrie', 'Portugal', 'Danemark', 'Croatie', 'La République de Macédoine du Nord', 'Slovénie', 'Estonie', 'Finlande', 'Grèce', 'Türkiye', 'Pologne', 'Etats-Unis', 'Slovaquie', 'Allemagne', 'Monténégro', 'Italie', 'Norvège', 'Lituanie', 'France', 'Espagne', 'Islande', 'Lettonie', 'Albanie', 'Roumanie', 'Belgique'}
Standardized NATO members: {'Turkey', 'Canada', 'Luxembourg', 'Italy', 'Romania', 'Denmark', 'Portugal', 'Iceland', 'Bulgaria', 'North Macedonia', 'United States', 'Finland', 'Netherlands', 'Croatie', 'Slovakia', 'Slovenia', 'Estonia', 'Hungary', 'Norway', 'Montenegro', 'Sweden', 'Spain', 'Czechia', 'Germany', 'Belgium', 'Latvia', 'France', 'Greece', 'Albanie', 'Poland', 'United Kingdom', 'Lithuania'}
Updated 'alliance' column for first 5 rows:
         Country alliance
0  United States     NATO
1         Russia    Other
2          China    Other
3          Indi

## Final Data Merge and Save

### Subtask:
Save the updated `dataset` DataFrame to `military_raw_data.csv`.


In [47]:
dataset.to_csv("military_raw_data.csv", index=False)
print("military_raw_data.csv generated successfully with updated data.")

military_raw_data.csv generated successfully with updated data.


In [50]:
import pandas as pd
from pathlib import Path


def summarize_csv_shape(csv_path: str) -> None:
    file = Path("/content/military_raw_data.csv")

    if not file.exists():
        print("CSV file not found.")
        return

    df = pd.read_csv(file)

    row_count, column_count = df.shape

    print(f"File analyzed : {file.name}")
    print(f"Total rows    : {row_count}")
    print(f"Total columns : {column_count}")


if __name__ == "__main__":
    summarize_csv_shape("/content/military_raw_data.csv")


File analyzed : military_raw_data.csv
Total rows    : 145
Total columns : 61
